In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

/home/teaching/miniconda3/envs/dl45/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "models/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
)
model = model.to("cuda")
model.config.use_cache = False            


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.80it/s]


In [3]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438


In [5]:
dataset = load_dataset("json", data_files={
    "train": "dataset/edit/train.json",
    "validation": "dataset/edit/val.json"
})

In [6]:
def edit_prompt(current_json, instruction):
    return f"""
### SYSTEM:
You are an AI system designed to MODIFY an existing job description JSON.

Your task:
- Update the given JSON based ONLY on the user instruction.
- Do NOT regenerate the entire job description.
- Make ONLY the necessary changes.

---

RULES:
- Output MUST be valid JSON only.
- Return ONLY ONE JSON object.
- Do NOT include explanations or extra text.
- Do NOT change fields that are not related to the instruction.
- Preserve all existing data unless modification is required.
- Do NOT hallucinate new fields or unnecessary content.

---

EDITING GUIDELINES:

1. ADD:
- Add new items to the correct field without removing existing ones.

2. REMOVE:
- Remove only the specified content.
- Do NOT delete unrelated items.

3. UPDATE:
- Modify only the specified field value.

4. REPLACE:
- Replace only the mentioned parts.

5. REFINE:
- Improve wording while preserving meaning.

6. IMPROVE:
- Make content more professional or detailed without changing intent.

7. REGENERATE:
- Rewrite the entire JSON ONLY if explicitly requested.

---

### USER:

Existing JSON:
{current_json}

Instruction:
{instruction}

---

### RESPONSE:
"""

In [7]:
def format_example(input_text):
    try:
        parts = input_text["input"].split("Instruction:\n")

        json_part = parts[0].replace("Existing JSON:\n", "").strip()
        instruction = parts[1].strip()

        prompt = edit_prompt(json_part, instruction)
        full_text = prompt + "\n" + input_text["output"]
        return {"text": full_text}

    except Exception as e:
        print("Error:", e)
        return {"text": ""}

In [8]:
dataset = dataset.map(format_example)
dataset = dataset.filter(lambda x: x["text"] is not None and len(x["text"]) > 0)

In [9]:
def tokenize(example):
    full_text = example["text"]

    tokens = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=320
    )

    labels = tokens["input_ids"].copy()

    # 🔥 Find where response starts
    response_start = full_text.find("### RESPONSE:")

    if response_start != -1:
        prefix = full_text[:response_start]
        prefix_tokens = tokenizer(prefix, truncation=True, max_length=256)["input_ids"]

        # 🔥 mask prompt part
        for i in range(len(prefix_tokens)):
            labels[i] = -100

    tokens["labels"] = labels
    return tokens

In [10]:
dataset = dataset.map(
    tokenize,
    remove_columns=dataset["train"].column_names
)

In [15]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./edit_model_results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   # 🔥 avoids OOM
    num_train_epochs=1.5,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

In [17]:
trainer.train()

                                                
  7%|▋         | 10/143 [00:11<02:35,  1.17s/it]

{'loss': 9.7587, 'grad_norm': 8.895689010620117, 'learning_rate': 9.370629370629372e-05, 'epoch': 0.11}


                                                
 14%|█▍        | 20/143 [00:23<02:22,  1.16s/it]

{'loss': 5.7274, 'grad_norm': 9.741704940795898, 'learning_rate': 8.741258741258743e-05, 'epoch': 0.21}


                                                
 21%|██        | 30/143 [00:35<02:13,  1.18s/it]

...

                                                 
 98%|█████████▊| 140/143 [02:48<00:03,  1.23s/it]

{'loss': 3.9101, 'grad_norm': 10.31314468383789, 'learning_rate': 3.496503496503497e-06, 'epoch': 1.46}


                                                 
100%|██████████| 143/143 [02:52<00:00,  1.20s/it]

{'train_runtime': 172.3079, 'train_samples_per_second': 6.625, 'train_steps_per_second': 0.83, 'train_loss': 4.695121204936421, 'epoch': 1.49}


TrainOutput(global_step=143, training_loss=4.695121204936421, metrics={'train_runtime': 172.3079, 'train_samples_per_second': 6.625, 'train_steps_per_second': 0.83, 'total_flos': 4433552631889920.0, 'train_loss': 4.695121204936421, 'epoch': 1.4940867279894876})

In [18]:
model.save_pretrained("./models/gemma-2b-it-fine-tuned-edit")
tokenizer.save_pretrained("./models/gemma-2b-it-fine-tuned-edit")

('./models/gemma-2b-it-fine-tuned-edit/tokenizer_config.json',
 './models/gemma-2b-it-fine-tuned-edit/special_tokens_map.json',
 './models/gemma-2b-it-fine-tuned-edit/tokenizer.json')